# 진열+전단 결합 프로모션 강건성 검증

**가설**: 동일 상품·매장에서 진열과 전단을 함께 적용한 주차는 단독 적용 주차보다 판매발생률과 매출이 높다.

이 노트북은 기존 결과가 분석 조건에 우연히 좌우되지 않는지 확인한다.

1. 매칭 허용 범위를 ±1주, ±2주, ±4주로 바꾼다.
2. 판매발생률뿐 아니라 매출, 수량, 장바구니 수도 확인한다.
3. 상품 수가 많은 상위 부문을 하나씩 제외해도 결론이 유지되는지 확인한다.

> 주의: 관찰자료의 동일 상품·매장 근접주차 비교이므로 무작위실험 수준의 인과효과로 단정하지 않는다.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
TRANSACTION_PATH = PROJECT_DIR / "transaction_data.csv"
CAUSAL_PATH = PROJECT_DIR / "causal_data.csv"
PRODUCT_PATH = PROJECT_DIR / "product.csv"
CHUNK_SIZE = 1_000_000
TOLERANCES = (1, 2, 4)
OUTCOMES = ["has_sale", "revenue", "units", "baskets"]
KEY_COLS = ["PRODUCT_ID", "STORE_ID", "WEEK_NO"]


## 1. 상품·매장·주차별 실제 판매 실적 준비

In [ ]:
tx = pd.read_csv(
    TRANSACTION_PATH,
    usecols=["BASKET_ID", "PRODUCT_ID", "STORE_ID", "WEEK_NO", "QUANTITY", "SALES_VALUE"],
    dtype={"BASKET_ID": "int64", "PRODUCT_ID": "int32", "STORE_ID": "int32",
           "WEEK_NO": "int16", "QUANTITY": "int32", "SALES_VALUE": "float32"},
)
tx_week = tx.groupby(KEY_COLS).agg(
    revenue=("SALES_VALUE", "sum"),
    units=("QUANTITY", "sum"),
    baskets=("BASKET_ID", "nunique"),
)
print(f"판매가 발생한 상품×매장×주차: {len(tx_week):,}")


## 2. 가장 가까운 단독 프로모션 주차와 매칭

±4주 안의 가장 가까운 단독 프로모션 주차를 먼저 찾은 뒤, 실제 주차 차이가 1·2·4 이하인 표본으로 각각 제한한다. 따라서 각 허용 범위는 같은 규칙을 사용하면서 표본의 시간적 근접성만 달라진다. 판매가 없던 주차는 0으로 유지한다.


In [ ]:
def pair_rows(data, single_group, comparison_name):
    combo = data.loc[data["promo_group"].eq("진열+전단"), KEY_COLS + OUTCOMES].copy()
    single = data.loc[data["promo_group"].eq(single_group), KEY_COLS + OUTCOMES].copy()
    if combo.empty or single.empty:
        return pd.DataFrame()
    single["matched_week"] = single["WEEK_NO"]
    single = single.rename(columns={c: f"{c}_single" for c in OUTCOMES})
    pairs = pd.merge_asof(
        combo.sort_values("WEEK_NO"), single.sort_values("WEEK_NO"),
        on="WEEK_NO", by=["PRODUCT_ID", "STORE_ID"], direction="nearest", tolerance=4,
    ).dropna(subset=["matched_week"])
    if pairs.empty:
        return pairs
    pairs["week_distance"] = (pairs["WEEK_NO"] - pairs["matched_week"]).abs()
    pairs["comparison"] = comparison_name
    for outcome in OUTCOMES:
        pairs[f"{outcome}_diff"] = pairs[outcome] - pairs[f"{outcome}_single"]
    return pairs


def aggregate_product_effects(pairs, tolerance):
    pairs = pairs.loc[pairs["week_distance"].le(tolerance)].copy()
    if pairs.empty:
        return pd.DataFrame()
    store = pairs.groupby(["PRODUCT_ID", "STORE_ID", "comparison"]).agg(
        matched_pairs=("WEEK_NO", "size"),
        mean_week_distance=("week_distance", "mean"),
        combo_sales_incidence=("has_sale", "mean"),
        single_sales_incidence=("has_sale_single", "mean"),
        combo_revenue=("revenue", "mean"),
        single_revenue=("revenue_single", "mean"),
        sales_incidence_diff=("has_sale_diff", "mean"),
        revenue_diff=("revenue_diff", "mean"),
        units_diff=("units_diff", "mean"),
        baskets_diff=("baskets_diff", "mean"),
    ).reset_index()
    product = store.groupby(["PRODUCT_ID", "comparison"]).agg(
        product_stores=("STORE_ID", "nunique"), matched_pairs=("matched_pairs", "sum"),
        mean_week_distance=("mean_week_distance", "mean"),
        combo_sales_incidence=("combo_sales_incidence", "mean"),
        single_sales_incidence=("single_sales_incidence", "mean"),
        combo_revenue=("combo_revenue", "mean"), single_revenue=("single_revenue", "mean"),
        sales_incidence_diff=("sales_incidence_diff", "mean"), revenue_diff=("revenue_diff", "mean"),
        units_diff=("units_diff", "mean"), baskets_diff=("baskets_diff", "mean"),
    ).reset_index()
    product["tolerance_weeks"] = tolerance
    return product


def process_complete_products(causal):
    causal["display_flag"] = causal["display"].fillna("0").ne("0")
    causal["mailer_flag"] = causal["mailer"].fillna("0").ne("0")
    causal["promo_group"] = np.select(
        [causal["display_flag"] & causal["mailer_flag"], causal["display_flag"], causal["mailer_flag"]],
        ["진열+전단", "진열만", "전단만"], default="확인 필요",
    )
    causal = causal.join(tx_week, on=KEY_COLS)
    for c in ["revenue", "units", "baskets"]:
        causal[c] = causal[c].fillna(0)
    causal["has_sale"] = causal["revenue"].gt(0).astype(int)
    outputs = []
    for single, name in [("진열만", "진열+전단 - 진열만"), ("전단만", "진열+전단 - 전단만")]:
        pairs = pair_rows(causal, single, name)
        if not pairs.empty:
            for tolerance in TOLERANCES:
                result = aggregate_product_effects(pairs, tolerance)
                if not result.empty:
                    outputs.append(result)
    return outputs


In [ ]:
effect_parts = []
carry = pd.DataFrame()
previous_last_product = None
dtype = {"PRODUCT_ID": "int32", "STORE_ID": "int32", "WEEK_NO": "int16",
         "display": "string", "mailer": "string"}
for chunk_no, chunk in enumerate(pd.read_csv(CAUSAL_PATH, chunksize=CHUNK_SIZE, dtype=dtype), start=1):
    if previous_last_product is not None and chunk["PRODUCT_ID"].iloc[0] < previous_last_product:
        raise ValueError("causal_data가 PRODUCT_ID 순으로 정렬돼 있지 않습니다.")
    previous_last_product = int(chunk["PRODUCT_ID"].iloc[-1])
    if not carry.empty:
        chunk = pd.concat([carry, chunk], ignore_index=True)
    last_product = chunk["PRODUCT_ID"].iloc[-1]
    carry = chunk.loc[chunk["PRODUCT_ID"].eq(last_product)].copy()
    complete = chunk.loc[~chunk["PRODUCT_ID"].eq(last_product)].copy()
    if not complete.empty:
        effect_parts.extend(process_complete_products(complete))
    if chunk_no % 10 == 0:
        print(f"{chunk_no}개 청크 처리 완료")
effect_parts.extend(process_complete_products(carry))
promotion_robustness_product_effects = pd.concat(effect_parts, ignore_index=True)
print(f"상품 단위 효과 행: {len(promotion_robustness_product_effects):,}")


## 3. 허용 범위별 가설 검정

In [ ]:
def normal_test(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n < 2:
        return {"products": n, "mean_effect": np.nan, "ci_low": np.nan, "ci_high": np.nan,
                "p_value_one_sided": np.nan}
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(n)
    z = mean / se if se > 0 else (np.inf if mean > 0 else 0)
    p_value = 0.5 * math.erfc(z / math.sqrt(2))
    return {"products": n, "mean_effect": mean, "ci_low": mean - 1.96 * se,
            "ci_high": mean + 1.96 * se, "p_value_one_sided": p_value}

rows = []
outcome_map = [("판매발생률", "sales_incidence_diff"), ("매출", "revenue_diff"),
               ("수량", "units_diff"), ("장바구니 수", "baskets_diff")]
for (tolerance, comparison), group in promotion_robustness_product_effects.groupby(["tolerance_weeks", "comparison"]):
    for outcome, column in outcome_map:
        result = normal_test(group[column])
        rows.append({"tolerance_weeks": tolerance, "comparison": comparison, "outcome": outcome, **result})
promotion_robustness_tests = pd.DataFrame(rows)
promotion_robustness_tests["significant_positive"] = (
    promotion_robustness_tests["mean_effect"].gt(0) & promotion_robustness_tests["p_value_one_sided"].lt(0.05)
)
display(promotion_robustness_tests)


## 4. 상위 대형 부문 제외 민감도

In [ ]:
products = pd.read_csv(PRODUCT_PATH, usecols=["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"])
product_effects_cat = promotion_robustness_product_effects.merge(products, on="PRODUCT_ID", how="left", validate="many_to_one")
department_sizes = products.groupby("DEPARTMENT")["PRODUCT_ID"].nunique().sort_values(ascending=False)
top_departments = department_sizes.head(5).index.tolist()
exclusion_rows = []
for (tolerance, comparison), group in product_effects_cat.groupby(["tolerance_weeks", "comparison"]):
    scenarios = [("전체", group)] + [(f"{department} 제외", group.loc[group["DEPARTMENT"].ne(department)])
                                      for department in top_departments]
    for scenario, sample in scenarios:
        for outcome, column in [("판매발생률", "sales_incidence_diff"), ("매출", "revenue_diff")]:
            result = normal_test(sample[column])
            exclusion_rows.append({"tolerance_weeks": tolerance, "comparison": comparison,
                                   "scenario": scenario, "outcome": outcome, **result})
promotion_department_exclusion_tests = pd.DataFrame(exclusion_rows)
promotion_department_exclusion_tests["significant_positive"] = (
    promotion_department_exclusion_tests["mean_effect"].gt(0) &
    promotion_department_exclusion_tests["p_value_one_sided"].lt(0.05)
)
print("제외 민감도 대상 부문:", ", ".join(top_departments))
display(promotion_department_exclusion_tests)


## 5. 결론 및 저장

In [ ]:
primary = promotion_robustness_tests.query("outcome in ['판매발생률', '매출']")
all_windows_pass = len(primary) == 12 and primary["significant_positive"].all()
exclusion_pass = promotion_department_exclusion_tests["significant_positive"].all()
print("모든 매칭 범위의 핵심 지표 통과:", all_windows_pass)
print("상위 5개 부문 개별 제외 후에도 통과:", exclusion_pass)
if all_windows_pass and exclusion_pass:
    print("최종 판정: 진열+전단 결합 효과는 매칭 범위와 대형 부문에 비교적 강건하다.")
else:
    print("최종 판정: 일부 조건에서 결론이 달라지므로 적용 범위를 제한해 해석해야 한다.")

promotion_robustness_product_effects.to_csv(
    OUTPUT_DIR / "promotion_robustness_product_effects.csv", index=False, encoding="utf-8-sig")
promotion_robustness_tests.to_csv(
    OUTPUT_DIR / "promotion_robustness_tests.csv", index=False, encoding="utf-8-sig")
promotion_department_exclusion_tests.to_csv(
    OUTPUT_DIR / "promotion_department_exclusion_tests.csv", index=False, encoding="utf-8-sig")
print("강건성 검증 결과 3개 저장 완료")
